# Step 1: Parse Dota 2 Replay Files (.dem → combatlog)

**Paper:** Behavioral Transparency in Multi-Agent RL: Strategic Decoupling and the Economic Optimization of Tactical Shocks  
**Author:** Kenny Ching — School of Business, University of Auckland  

---

### Description
This notebook parses the raw Dota 2 replay files (`.dem`) for the three publicly released
OpenAI Five matches using the open-source **Clarity** Java parser. It produces a structured
combat log text file for each game, which is consumed by Step 2.

### Input
Upload the three `.dem` replay files when prompted:
- `og_game_1_04132013569395266.dem` — OG vs OpenAI Five, Game 1 (OG wins)
- `og_game2_04132030946085141.dem` — OG vs OpenAI Five, Game 2 (OpenAI wins)
- `coop_game_04132076146922985.dem` — OpenAI Five Coop Game

Source: https://openai.com/blog/openai-five-finals

### Output
- `og_game1_combatlog.txt`
- `og_game2_combatlog.txt`
- `coop_combatlog.txt`

### Requirements
- Java 17+ (Colab has this by default)
- Git (Colab has this by default)
- Internet access to clone Clarity and download Gradle dependencies (~2 min first run)

In [ ]:
# ============================================================
# CELL 1 — Check Java version and upload .dem files
# ============================================================
import subprocess, os
from google.colab import files

# Verify Java
result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print('Java:', result.stderr.strip().split('\n')[0])

# Upload dem files
print('\nPlease upload all 3 .dem files:')
uploaded = files.upload()

DEM_FILES = {}
for fname in uploaded:
    dest = f'/content/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    DEM_FILES[fname] = dest
    size_mb = os.path.getsize(dest) / 1024 / 1024
    print(f'  {fname}: {size_mb:.1f} MB')

print(f'\nUploaded {len(DEM_FILES)} file(s).')

In [ ]:
# ============================================================
# CELL 2 — Clone and build Clarity combatlog parser
# ============================================================
import subprocess

CLARITY_DIR = '/content/clarity-examples'
JAR_PATH    = f'{CLARITY_DIR}/build/libs/combatlog.jar'

if not os.path.exists(JAR_PATH):
    print('Cloning clarity-examples...')
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/skadistats/clarity-examples.git', CLARITY_DIR],
        check=True
    )

    # Patch build file: set Java version to 21 (Colab default)
    build_file = f'{CLARITY_DIR}/build.gradle.kts'
    with open(build_file) as f:
        content = f.read()
    content = content.replace(
        'languageVersion.set(JavaLanguageVersion.of(17))',
        'languageVersion.set(JavaLanguageVersion.of(21))'
    )
    with open(build_file, 'w') as f:
        f.write(content)
    print('Patched build.gradle.kts for Java 21')

    print('Building combatlog jar (this takes ~2 minutes)...')
    result = subprocess.run(
        ['./gradlew', 'combatlogPackage'],
        cwd=CLARITY_DIR,
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('BUILD FAILED:')
        print(result.stderr[-3000:])
    else:
        print('Build successful.')
else:
    print(f'JAR already exists: {JAR_PATH}')

print(f'JAR exists: {os.path.exists(JAR_PATH)}')

In [ ]:
# ============================================================
# CELL 3 — Run Clarity on all .dem files
# ============================================================
import subprocess, os

# Map uploaded filenames to output labels
# Adjust if your filenames differ
GAME_MAP = {
    'og_game1':  None,  # will be auto-detected below
    'og_game2':  None,
    'coop':      None,
}

# Auto-detect which dem file is which by filename
for fname, path in DEM_FILES.items():
    f = fname.lower()
    if 'game_1' in f or 'game1' in f or '13569' in f:
        GAME_MAP['og_game1'] = path
    elif 'game2' in f or 'game_2' in f or '30946' in f:
        GAME_MAP['og_game2'] = path
    elif 'coop' in f or '76146' in f:
        GAME_MAP['coop'] = path

print('Detected dem files:')
for label, path in GAME_MAP.items():
    print(f'  {label}: {path}')

missing = [k for k,v in GAME_MAP.items() if v is None]
if missing:
    print(f'\nWARNING: Could not auto-detect: {missing}')
    print('Manually assign paths in GAME_MAP before continuing.')
else:
    OUTPUT_DIR = '/content/combatlog'
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for label, dem_path in GAME_MAP.items():
        out_path = f'{OUTPUT_DIR}/{label}_combatlog.txt'
        print(f'\nParsing {label}...')
        result = subprocess.run(
            ['java', '-jar', JAR_PATH, dem_path],
            capture_output=True, text=True, timeout=300
        )
        with open(out_path, 'w') as f:
            f.write(result.stdout)
        lines = result.stdout.count('\n')
        size_kb = os.path.getsize(out_path) // 1024
        print(f'  -> {out_path}  ({lines:,} lines, {size_kb} KB)')

    print('\n✅ All combatlog files generated.')

In [ ]:
# ============================================================
# CELL 4 — Verify output (quick sanity check)
# ============================================================
import re

for label in ['og_game1', 'og_game2', 'coop']:
    path = f'/content/combatlog/{label}_combatlog.txt'
    if not os.path.exists(path):
        print(f'{label}: FILE MISSING'); continue

    with open(path) as f:
        lines = f.readlines()

    hero_kills  = sum(1 for l in lines if 'is killed by npc_dota_hero' in l and 'npc_dota_hero' in l.split('is killed')[0])
    gold_events = sum(1 for l in lines if 'receives' in l and 'gold' in l)
    state4      = next((l.strip() for l in lines if 'game state is now 4' in l), 'not found')
    state6      = next((l.strip() for l in lines if 'game state is now 6' in l), 'not found')

    print(f'\n{label}:')
    print(f'  Lines:       {len(lines):,}')
    print(f'  Hero kills:  {hero_kills}')
    print(f'  Gold events: {gold_events}')
    print(f'  Game start:  {state4}')
    print(f'  Game end:    {state6}')

print('\n✅ Verification complete. Proceed to Step 2.')

In [ ]:
# ============================================================
# CELL 5 — Download combatlog files
# ============================================================
import zipfile
from google.colab import files

zip_path = '/content/combatlog_files.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for label in ['og_game1', 'og_game2', 'coop']:
        p = f'/content/combatlog/{label}_combatlog.txt'
        if os.path.exists(p):
            zf.write(p, f'{label}_combatlog.txt')

files.download(zip_path)
print('Downloaded combatlog_files.zip')
print('Upload these txt files when running Step 2.')